## Example 1: points aggregated onto a pixel grid

In [ ]:
import numpy as np
import pandas as pd
import datashader as ds

# Example 1: Canvas.points bins every point into one pixel; the default reduction counts them.
print("Example 1: points aggregated onto a pixel grid")
rng = np.random.default_rng(0)
n = 10_000
df = pd.DataFrame({"x": rng.uniform(0, 1, n), "y": rng.uniform(0, 1, n)})

canvas = ds.Canvas(plot_width=100, plot_height=80, x_range=(0, 1), y_range=(0, 1))
agg = canvas.points(df, "x", "y")
assert agg.dims == ("y", "x") and agg.shape == (80, 100), (agg.dims, agg.shape)
assert int(agg.sum()) == n, int(agg.sum())  # every point lands in exactly one pixel
assert np.isclose(float(agg.coords["x"][0]), 0.005) and np.isclose(float(agg.coords["y"][0]), 0.5 / 80)  # pixel centres
print(f"  {n} points -> {agg.shape} grid of {agg.dtype}, busiest pixel holds {int(agg.max())}")
print("Example 1 finished")

## Example 2: sum, max and mean reductions, and points outside the canvas

In [ ]:
import numpy as np
import pandas as pd
import datashader as ds

# Example 2: other reductions over a value column; points outside x_range/y_range are dropped.
print("Example 2: sum, max and mean reductions")
rng = np.random.default_rng(1)
n = 5_000
df = pd.DataFrame({"x": rng.uniform(0, 1, n), "y": rng.uniform(0, 1, n), "v": rng.uniform(0, 2, n)})
canvas = ds.Canvas(plot_width=40, plot_height=40, x_range=(0, 1), y_range=(0, 1))

total = canvas.points(df, "x", "y", agg=ds.sum("v"))
assert np.isclose(float(total.sum()), df["v"].sum()), (float(total.sum()), df["v"].sum())
peak = canvas.points(df, "x", "y", agg=ds.max("v"))
assert np.isclose(float(peak.max()), df["v"].max()), (float(peak.max()), df["v"].max())

count = canvas.points(df, "x", "y", agg=ds.count())
mean = canvas.points(df, "x", "y", agg=ds.mean("v"))
filled = count.values > 0
assert np.allclose(mean.values[filled], total.values[filled] / count.values[filled])
assert np.isnan(mean.values[~filled]).all()  # an empty pixel has no mean
print(f"  sum {float(total.sum()):.3f}, max {float(peak.max()):.3f}, {int(filled.sum())} of 1600 pixels filled")

outside = canvas.points(df.assign(x=df["x"] + 2), "x", "y")
assert int(outside.sum()) == 0, int(outside.sum())
print("  points shifted off the canvas are not counted")
print("Example 2 finished")

## Example 3: shading an aggregate into an RGBA image

In [ ]:
import numpy as np
import pandas as pd
import datashader as ds
import datashader.transfer_functions as tf

# Example 3: tf.shade maps an aggregate to packed RGBA pixels; empty pixels stay transparent.
print("Example 3: shading an aggregate into an RGBA image")
rng = np.random.default_rng(2)
df = pd.DataFrame({"x": rng.normal(0.5, 0.15, 20_000), "y": rng.normal(0.5, 0.15, 20_000)})
agg = ds.Canvas(plot_width=100, plot_height=80, x_range=(0, 1), y_range=(0, 1)).points(df, "x", "y")

img = tf.shade(agg, cmap=["lightblue", "darkblue"], how="linear")
assert isinstance(img, tf.Image) and img.shape == agg.shape and img.dtype == np.uint32, (type(img), img.shape, img.dtype)
rgba = img.data.view(np.uint8).reshape(img.shape + (4,))
assert ((rgba[..., 3] == 0) == (agg.values == 0)).all(), "transparent pixels should be exactly the empty ones"

opaque = tf.set_background(img, "white")
assert (opaque.data.view(np.uint8).reshape(img.shape + (4,))[..., 3] == 255).all()
spread = tf.spread(img, px=1)
assert spread.shape == img.shape
print(f"  {img.shape} image, {int((rgba[..., 3] == 0).sum())} transparent pixels; set_background and spread keep the shape")
print("Example 3 finished")

## Example 4: categorical counts with count_cat

In [ ]:
import numpy as np
import pandas as pd
import datashader as ds

# Example 4: count_cat adds a category axis, one count per category per pixel.
print("Example 4: categorical counts with count_cat")
rng = np.random.default_rng(3)
n = 9_000
df = pd.DataFrame({
    "x": rng.uniform(0, 1, n),
    "y": rng.uniform(0, 1, n),
    "kind": pd.Categorical(rng.choice(["a", "b", "c"], n)),
})
agg = ds.Canvas(plot_width=50, plot_height=50, x_range=(0, 1), y_range=(0, 1)).points(df, "x", "y", agg=ds.count_cat("kind"))
assert agg.dims == ("y", "x", "kind") and agg.shape == (50, 50, 3), (agg.dims, agg.shape)
assert list(agg.coords["kind"].values) == ["a", "b", "c"]
per_kind = agg.sum(dim=("y", "x")).to_series().to_dict()
expected = df["kind"].value_counts().to_dict()
assert per_kind == expected, (per_kind, expected)
print(f"  per-category totals match pandas: {per_kind}")
print("Example 4 finished")

## Example 5: lines and areas

In [ ]:
import numpy as np
import pandas as pd
import datashader as ds

# Example 5: Canvas.line rasterises the segments between points; Canvas.area fills down to y=0.
print("Example 5: lines and areas")
canvas = ds.Canvas(plot_width=50, plot_height=50, x_range=(0, 1), y_range=(0, 1))
diagonal = pd.DataFrame({"x": np.linspace(0, 1, 500), "y": np.linspace(0, 1, 500)})
line = canvas.line(diagonal, "x", "y", agg=ds.any())
assert np.array_equal(line.values, np.eye(50, dtype=bool)), "the diagonal line should cover exactly the diagonal pixels"
print(f"  line from (0, 0) to (1, 1) covers {int(line.sum())} pixels, all on the diagonal")

level = pd.DataFrame({"x": np.linspace(0, 1, 101), "y": np.full(101, 0.45)})
area = ds.Canvas(plot_width=10, plot_height=10, x_range=(0, 1), y_range=(0, 1)).area(level, "x", "y", agg=ds.any())
assert area.values[:5].all() and not area.values[5:].any(), area.values.astype(int)
print("  area under y=0.45 fills the bottom 5 of 10 pixel rows and nothing above")
print("Example 5 finished")